# PatchTST Base Model — Delhi Air Quality Forecasting (Anand Vihar)

**Purpose:** Train a Patch Time Series Transformer (PatchTST) to forecast the next 24 hours of all 11 pollutant measurements from the previous 24 hours.

| Setting | Value |
|---|---|
| Dataset | Combined Anand Vihar 2025 + Jan–Aug 2026 hourly data |
| Inputs | 24 historical hours × 11 pollutants |
| Outputs | 24 future hours × 11 pollutants |
| Split | Chronological 70% train / 15% validation / 15% test |
| Scaling | `RobustScaler`, fitted on training data only |
| Missing inputs | Imputed with training-period pollutant medians |
| Missing targets | Excluded from loss and evaluation using a mask |
| Framework | PyTorch (Google Colab) |

**Important evaluation convention:** this notebook evaluates rolling forecast origins. Every test forecast predicts a 24-hour block entirely inside the test period, using the 24 hours immediately before that forecast origin as historical input. This permits the most recent *observed* readings to be used as the forecast origin advances, which is a realistic hourly-updated forecasting setup. No target from a validation or test forecast window is used as its input.

**Before running:** In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). Upload the combined CSV linked with this notebook, not the two source files separately.

## 1. Install / import libraries

In [ ]:
# Colab already includes PyTorch in most runtimes. Install only the data/plotting packages if needed.
%pip -q install numpy pandas scikit-learn matplotlib

import os, glob, json, time, copy, random, zipfile, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Reproducibility and experiment configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Keep deterministic behaviour where supported. This may reduce speed slightly.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TIMESTAMP_COL = "Timestamp"
FEATURES = [
    "PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO (µg/m³)", "NO2 (µg/m³)",
    "NOx (ppb)", "NH3 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)",
    "Ozone (µg/m³)", "Benzene (µg/m³)", "Toluene (µg/m³)"
]

SEQ_LEN = 24
HORIZON = 24
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 1.0

# PatchTST configuration. Each pollutant is processed as an independent
# univariate channel using shared Transformer weights (channel independence).
# With a 24-hour input, patch_len=4 and stride=2 produce 11 overlapping patches.
PATCH_LEN = 4
PATCH_STRIDE = 2
D_MODEL = 64
N_HEADS = 4
E_LAYERS = 3
D_FF = 128
DROPOUT = 0.10

assert D_MODEL % N_HEADS == 0, "D_MODEL must be divisible by N_HEADS."
assert PATCH_LEN <= SEQ_LEN, "PATCH_LEN cannot exceed SEQ_LEN."
NUM_PATCHES = 1 + (SEQ_LEN - PATCH_LEN) // PATCH_STRIDE

print("Device:", DEVICE)
print("Input shape per sample:", (SEQ_LEN, len(FEATURES)))
print("Output shape per sample:", (HORIZON, len(FEATURES)))
print("Patch configuration:", {
    "patch_len": PATCH_LEN, "patch_stride": PATCH_STRIDE,
    "number_of_patches": NUM_PATCHES, "d_model": D_MODEL,
    "heads": N_HEADS, "encoder_layers": E_LAYERS, "feedforward_dim": D_FF
})


## 3. Upload and load the combined dataset

In [ ]:
EXPECTED_FILE = "Delhi_AnandVihar_2025_Aug2026_11pollutants_combined_unimputed.csv"
CSV_PATH = ""  # Optional: set this to a file path if the CSV is already in the runtime.

if not CSV_PATH:
    try:
        from google.colab import files
        print("Upload the combined CSV file:")
        uploaded = files.upload()
        csv_candidates = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if EXPECTED_FILE in csv_candidates:
            CSV_PATH = EXPECTED_FILE
        elif len(csv_candidates) == 1:
            CSV_PATH = csv_candidates[0]
        else:
            raise ValueError(
                "Please upload exactly the combined 11-pollutant CSV. "
                f"CSV files received: {csv_candidates}"
            )
    except ImportError:
        candidates = glob.glob("*.csv") + glob.glob("/mnt/data/*.csv")
        preferred = [p for p in candidates if Path(p).name == EXPECTED_FILE]
        if preferred:
            CSV_PATH = preferred[0]
        elif len(candidates) == 1:
            CSV_PATH = candidates[0]
        else:
            raise FileNotFoundError(
                "Set CSV_PATH to the combined CSV location. Candidates found: " + str(candidates)
            )

print("Using:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
if TIMESTAMP_COL not in df.columns:
    raise ValueError(f"Missing required timestamp column: {TIMESTAMP_COL}")
missing_features = [c for c in FEATURES if c not in df.columns]
if missing_features:
    raise ValueError(f"Required pollutant columns are missing: {missing_features}")

# Keep only timestamp + the agreed 11 pollutant features, and sort chronologically.
df = df[[TIMESTAMP_COL] + FEATURES].copy()
df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors="raise")
df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

if df[TIMESTAMP_COL].duplicated().any():
    dupes = df.loc[df[TIMESTAMP_COL].duplicated(keep=False), TIMESTAMP_COL]
    raise ValueError(f"Duplicate timestamps found. Resolve them before sequence creation.\n{dupes.head()}")

time_deltas = df[TIMESTAMP_COL].diff().dropna()
non_hourly = time_deltas.ne(pd.Timedelta(hours=1))
if non_hourly.any():
    examples = df.loc[non_hourly[non_hourly].index, TIMESTAMP_COL].head(10).tolist()
    raise ValueError(
        "The series is not strictly hourly. Do not silently compress time gaps. "
        f"Example timestamps after a non-hourly gap: {examples}"
    )

# Convert pollutant values to numeric. Unparseable cells become missing values.
for col in FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Shape:", df.shape)
print("First timestamp:", df[TIMESTAMP_COL].min())
print("Last timestamp:", df[TIMESTAMP_COL].max())
print("Duplicate timestamps:", int(df[TIMESTAMP_COL].duplicated().sum()))
display(df.head())

## 4. Quick data-quality report

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df[FEATURES].isna().sum(),
    "missing_percent": (df[FEATURES].isna().mean() * 100).round(2),
    "observed_count": df[FEATURES].notna().sum(),
})
display(missing_report)

complete_rows = int(df[FEATURES].notna().all(axis=1).sum())
print(f"Rows with all 11 pollutants observed: {complete_rows:,} / {len(df):,}")

# This is a diagnostic only. No values are automatically deleted or clipped.
print("\nTraining will preserve genuine high readings and mask missing targets.")
print("Investigate physically implausible sensor values against the original CPCB source separately.")

## 5. Chronological train / validation / test split

In [ ]:
N = len(df)
train_end = int(N * TRAIN_FRACTION)
val_end = int(N * (TRAIN_FRACTION + VAL_FRACTION))

# Split indices refer to row boundaries. Samples are constructed so that each target
# forecast block lies completely inside its own split. Historical context may look
# backwards across a boundary (e.g., the first validation input uses the final 24
# training hours), which is allowed because those readings are already historical.
train_start_time = df[TIMESTAMP_COL].iloc[0]
train_last_time = df[TIMESTAMP_COL].iloc[train_end - 1]
val_start_time = df[TIMESTAMP_COL].iloc[train_end]
val_last_time = df[TIMESTAMP_COL].iloc[val_end - 1]
test_start_time = df[TIMESTAMP_COL].iloc[val_end]
test_last_time = df[TIMESTAMP_COL].iloc[-1]

print(f"Total rows: {N:,}")
print(f"Train rows: {train_end:,} | {train_start_time} → {train_last_time}")
print(f"Validation rows: {val_end-train_end:,} | {val_start_time} → {val_last_time}")
print(f"Test rows: {N-val_end:,} | {test_start_time} → {test_last_time}")

assert train_end > SEQ_LEN + HORIZON
assert val_end - train_end >= HORIZON
assert N - val_end >= HORIZON

## 6. Fit preprocessing on training data only

In [ ]:
raw_values = df[FEATURES].to_numpy(dtype=np.float64)

# Medians are calculated only from the training period. They are used to fill missing
# historical input values, not to fabricate supervised target observations.
train_raw = raw_values[:train_end]
train_medians = np.nanmedian(train_raw, axis=0)
if not np.isfinite(train_medians).all():
    bad = [FEATURES[i] for i, v in enumerate(train_medians) if not np.isfinite(v)]
    raise ValueError(f"These features have no valid observations in the training split: {bad}")

input_missing_mask = ~np.isfinite(raw_values)
imputed_values = np.where(input_missing_mask, train_medians.reshape(1, -1), raw_values)

# Robust scaling is selected because pollutant data can be skewed and contain spikes.
# It is fitted on the imputed TRAINING period only. Use this exact preprocessing in
# the PatchTST notebook for a fair model-to-model comparison.
scaler = RobustScaler(quantile_range=(25.0, 75.0))
scaler.fit(imputed_values[:train_end])
scaled_values = scaler.transform(imputed_values).astype(np.float32)

# Targets are scaled with the same train-fitted scaler. Missing target positions are
# ignored via target_observed_mask during loss and metric calculation.
target_observed_mask = np.isfinite(raw_values)

print("Training medians:")
display(pd.Series(train_medians, index=FEATURES, name="training_median"))
print("Scaler center and scale are learned from training rows only.")

## 7. Create sliding 24-hour input / 24-hour target windows

In [ ]:
class AirQualityWindowDataset(Dataset):
    def __init__(self, values, observed_mask, target_starts, seq_len=24, horizon=24):
        self.values = values
        self.observed_mask = observed_mask
        self.target_starts = np.asarray(target_starts, dtype=np.int64)
        self.seq_len = seq_len
        self.horizon = horizon

    def __len__(self):
        return len(self.target_starts)

    def __getitem__(self, idx):
        target_start = int(self.target_starts[idx])
        input_start = target_start - self.seq_len
        x = self.values[input_start:target_start]                 # (24, 11)
        y = self.values[target_start:target_start+self.horizon]   # (24, 11)
        mask = self.observed_mask[target_start:target_start+self.horizon]
        return (
            torch.from_numpy(np.asarray(x, dtype=np.float32)),
            torch.from_numpy(np.asarray(y, dtype=np.float32)),
            torch.from_numpy(np.asarray(mask, dtype=np.bool_)),
        )

# Target starts are constrained so each target block stays inside its partition.
train_target_starts = np.arange(SEQ_LEN, train_end - HORIZON + 1)
val_target_starts = np.arange(train_end, val_end - HORIZON + 1)
test_target_starts = np.arange(val_end, N - HORIZON + 1)

train_dataset = AirQualityWindowDataset(scaled_values, target_observed_mask, train_target_starts, SEQ_LEN, HORIZON)
val_dataset = AirQualityWindowDataset(scaled_values, target_observed_mask, val_target_starts, SEQ_LEN, HORIZON)
test_dataset = AirQualityWindowDataset(scaled_values, target_observed_mask, test_target_starts, SEQ_LEN, HORIZON)

PIN_MEMORY = DEVICE.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)

print(f"Training windows:   {len(train_dataset):,}")
print(f"Validation windows: {len(val_dataset):,}")
print(f"Testing windows:    {len(test_dataset):,}")
print("Tensor shapes (one sample): X =", (SEQ_LEN, len(FEATURES)), ", y =", (HORIZON, len(FEATURES)))

# Every split must have observed future targets; missing entries are masked, not zeros.
for label, starts in [("train", train_target_starts), ("validation", val_target_starts), ("test", test_target_starts)]:
    observed = target_observed_mask[starts[:, None] + np.arange(HORIZON)[None, :]].sum()
    print(f"{label.title()} observed target cells: {observed:,}")

## 8. Define the PatchTST forecasting architecture

PatchTST divides each input time series into **overlapping temporal patches**, embeds those patches, and processes them with a Transformer encoder. This implementation follows the **channel-independent** design: each pollutant is processed as a separate univariate sequence, while the same patch embedding and Transformer weights are shared across all 11 pollutants. The shared model produces a 24-hour forecast for each channel, which are stacked into an `(batch, 24, 11)` output.

For the 24-hour input window, `patch_len=4` and `stride=2` create 11 overlapping patches. The Transformer uses `d_model=64`, 4 attention heads, 3 encoder layers and a feed-forward dimension of 128. These settings are deliberately moderate for a short input window and practical Colab training.

This is a baseline PatchTST implementation, not the proposed MA-PatchTST. The architecture and its hyperparameters are the only model-specific differences from the TCN notebook; data handling, splits, missing-value masks, training budget, evaluation metrics and hardware benchmarks are kept the same.


In [ ]:
class PatchTSTForecaster(nn.Module):
    """Channel-independent PatchTST for multivariate, multi-horizon forecasting.

    Input:  (batch, seq_len, n_features)
    Output: (batch, horizon, n_features)

    Each feature is treated as an independent univariate series. The patch
    projection and Transformer encoder weights are shared across all features.
    """
    def __init__(self, input_size, output_size, seq_len, horizon,
                 patch_len=4, patch_stride=2, d_model=64, n_heads=4,
                 e_layers=3, d_ff=128, dropout=0.10):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        if patch_len > seq_len:
            raise ValueError("patch_len cannot exceed seq_len")

        self.input_size = input_size
        self.output_size = output_size
        self.seq_len = seq_len
        self.horizon = horizon
        self.patch_len = patch_len
        self.patch_stride = patch_stride
        self.num_patches = 1 + (seq_len - patch_len) // patch_stride

        # Embed each raw patch into the Transformer latent dimension.
        self.patch_projection = nn.Linear(patch_len, d_model)
        self.position_embedding = nn.Parameter(
            torch.zeros(1, self.num_patches, d_model)
        )
        nn.init.trunc_normal_(self.position_embedding, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=e_layers,
            norm=nn.LayerNorm(d_model),
            enable_nested_tensor=False,
        )

        # Flatten encoded patches for each channel and map to the forecast horizon.
        self.forecast_head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(self.num_patches * d_model, horizon),
        )

    def forward(self, x):
        batch_size, time_steps, n_features = x.shape
        if time_steps != self.seq_len:
            raise ValueError(f"Expected {self.seq_len} input steps, got {time_steps}")
        if n_features != self.input_size:
            raise ValueError(f"Expected {self.input_size} features, got {n_features}")

        # Channel independence: reshape to (batch * features, time).
        x = x.transpose(1, 2).reshape(batch_size * n_features, time_steps)

        # (batch * features, number_of_patches, patch_len)
        patches = x.unfold(dimension=1, size=self.patch_len, step=self.patch_stride)
        z = self.patch_projection(patches) + self.position_embedding
        z = self.encoder(z)

        # Forecast each channel independently, then restore multivariate shape.
        forecast = self.forecast_head(z)  # (batch * features, horizon)
        forecast = forecast.reshape(batch_size, n_features, self.horizon)
        return forecast.transpose(1, 2).contiguous()  # (batch, horizon, features)


model = PatchTSTForecaster(
    input_size=len(FEATURES),
    output_size=len(FEATURES),
    seq_len=SEQ_LEN,
    horizon=HORIZON,
    patch_len=PATCH_LEN,
    patch_stride=PATCH_STRIDE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    e_layers=E_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
).to(DEVICE)

parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Trainable parameters: {parameter_count:,}")
print(f"Number of patches per channel: {NUM_PATCHES}")

# Shape check before the training run.
xb, yb, mb = next(iter(train_loader))
with torch.no_grad():
    check_output = model(xb[:2].to(DEVICE))
print("Shape check:", tuple(check_output.shape), "(expected: (2, 24, 11))")
assert tuple(check_output.shape) == (2, HORIZON, len(FEATURES))


## 9. Masked loss and training / validation functions

In [ ]:
def masked_mse_components(predictions, targets, observed_mask):
    """Return squared-error sum and valid-cell count; missing targets contribute nothing."""
    mask = observed_mask.to(dtype=predictions.dtype)
    squared_error = (predictions - targets).pow(2) * mask
    return squared_error.sum(), mask.sum()


@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    total_squared_error = 0.0
    total_observed = 0.0
    for xb, yb, mb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        mb = mb.to(DEVICE, non_blocking=True)
        predictions = model(xb)
        err_sum, count = masked_mse_components(predictions, yb, mb)
        total_squared_error += err_sum.item()
        total_observed += count.item()
    if total_observed == 0:
        raise RuntimeError("No observed target values were available for evaluation.")
    return total_squared_error / total_observed


def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-5
)

history = {"epoch": [], "train_masked_mse": [], "val_masked_mse": [], "learning_rate": []}
best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats(DEVICE)
sync_device()
training_start = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_squared_error = 0.0
    train_observed = 0.0

    for xb, yb, mb in train_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        mb = mb.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        predictions = model(xb)
        err_sum, observed_count = masked_mse_components(predictions, yb, mb)
        loss = err_sum / observed_count.clamp_min(1.0)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        train_squared_error += err_sum.detach().item()
        train_observed += observed_count.detach().item()

    train_loss = train_squared_error / max(train_observed, 1.0)
    val_loss = evaluate_loss(model, val_loader)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    history["epoch"].append(epoch)
    history["train_masked_mse"].append(train_loss)
    history["val_masked_mse"].append(val_loss)
    history["learning_rate"].append(current_lr)

    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | train masked MSE: {train_loss:.5f} | "
          f"val masked MSE: {val_loss:.5f} | lr: {current_lr:.2e}")

    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping: no validation improvement for {PATIENCE} epochs.")
        break

sync_device()
training_seconds = time.perf_counter() - training_start
if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")
model.load_state_dict(best_state)
model.eval()

print(f"\nTraining time: {training_seconds:.2f} seconds")
print(f"Best validation masked MSE: {best_val_loss:.6f}")
if DEVICE.type == "cuda":
    peak_gpu_memory_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2)
else:
    peak_gpu_memory_mb = float("nan")
print("Peak GPU memory (MB):", peak_gpu_memory_mb)

## 10. Training curves

In [ ]:
history_df = pd.DataFrame(history)
display(history_df.tail())

plt.figure(figsize=(9, 4.5))
plt.plot(history_df["epoch"], history_df["train_masked_mse"], label="Training")
plt.plot(history_df["epoch"], history_df["val_masked_mse"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Masked MSE (scaled target space)")
plt.title("PatchTST — Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("patchtst_training_validation_loss.png", dpi=160)
plt.show()

## 11. Generate predictions on the untouched test period

In [ ]:
@torch.no_grad()
def predict_scaled(model, loader):
    model.eval()
    predictions_list, targets_list, masks_list = [], [], []
    for xb, yb, mb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        predictions = model(xb).cpu().numpy()
        predictions_list.append(predictions)
        targets_list.append(yb.numpy())
        masks_list.append(mb.numpy())
    return (
        np.concatenate(predictions_list, axis=0),
        np.concatenate(targets_list, axis=0),
        np.concatenate(masks_list, axis=0),
    )

pred_scaled, true_scaled, test_mask = predict_scaled(model, test_loader)

# Inverse transform all feature values back to the original measurement units.
num_test_windows = pred_scaled.shape[0]
pred_original = scaler.inverse_transform(pred_scaled.reshape(-1, len(FEATURES))).reshape(
    num_test_windows, HORIZON, len(FEATURES)
)
true_original = scaler.inverse_transform(true_scaled.reshape(-1, len(FEATURES))).reshape(
    num_test_windows, HORIZON, len(FEATURES)
)

print("Prediction tensor:", pred_original.shape)
print("Observed target cells in test forecasts:", int(test_mask.sum()))
assert pred_original.shape == (len(test_dataset), HORIZON, len(FEATURES))

## 12. Test metrics — pollutant-wise and scale-aware summary

The pollutants have different units and numerical ranges (for example, CO is reported in mg/m³ while most others are in µg/m³). Therefore, **do not rank models using one pooled raw-unit MAE across all pollutants**. This section reports MAE, RMSE and R² separately for every pollutant, plus macro-averaged errors normalised by each pollutant's training-period IQR. Use the same definitions for PatchTST and every teammate's models.

In [ ]:
metric_rows = []
for j, feature in enumerate(FEATURES):
    valid = test_mask[:, :, j].astype(bool)
    y_true = true_original[:, :, j][valid]
    y_pred = pred_original[:, :, j][valid]
    if len(y_true) == 0:
        mae = rmse = r2 = np.nan
    else:
        mae = mean_absolute_error(y_true, y_pred)
        rmse = mean_squared_error(y_true, y_pred) ** 0.5
        r2 = r2_score(y_true, y_pred) if len(y_true) >= 2 and np.var(y_true) > 0 else np.nan
    feature_scale = float(scaler.scale_[j])
    metric_rows.append({
        "pollutant": feature,
        "observed_test_targets": int(valid.sum()),
        "MAE_original_units": mae,
        "RMSE_original_units": rmse,
        "R2": r2,
        "normalized_MAE_by_training_IQR": mae / feature_scale if np.isfinite(mae) else np.nan,
        "normalized_RMSE_by_training_IQR": rmse / feature_scale if np.isfinite(rmse) else np.nan,
        "training_IQR_scale": feature_scale,
    })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(4))

macro_summary = {
    "model": "PatchTST",
    "test_windows": int(num_test_windows),
    "observed_test_target_cells": int(test_mask.sum()),
    "macro_normalized_MAE_by_training_IQR": float(metrics_df["normalized_MAE_by_training_IQR"].mean()),
    "macro_normalized_RMSE_by_training_IQR": float(metrics_df["normalized_RMSE_by_training_IQR"].mean()),
    "macro_R2": float(metrics_df["R2"].mean()),
    "training_time_seconds": float(training_seconds),
    "trainable_parameters": int(parameter_count),
    "peak_GPU_memory_MB": float(peak_gpu_memory_mb) if np.isfinite(peak_gpu_memory_mb) else None,
}
print("Macro summary (use for the cross-model comparison, alongside pollutant-wise scores):")
display(pd.DataFrame([macro_summary]).round(5))

## 13. Error by forecast lead time (1–24 hours ahead)

In [ ]:
horizon_rows = []
for h in range(HORIZON):
    per_feature_nmae = []
    per_feature_nrmse = []
    per_feature_mae = []
    for j in range(len(FEATURES)):
        valid = test_mask[:, h, j].astype(bool)
        if valid.sum() == 0:
            continue
        yt = true_original[:, h, j][valid]
        yp = pred_original[:, h, j][valid]
        mae_h = mean_absolute_error(yt, yp)
        rmse_h = mean_squared_error(yt, yp) ** 0.5
        scale_h = float(scaler.scale_[j])
        per_feature_mae.append(mae_h)
        per_feature_nmae.append(mae_h / scale_h)
        per_feature_nrmse.append(rmse_h / scale_h)
    horizon_rows.append({
        "forecast_hour": h + 1,
        "macro_normalized_MAE_by_training_IQR": np.mean(per_feature_nmae) if per_feature_nmae else np.nan,
        "macro_normalized_RMSE_by_training_IQR": np.mean(per_feature_nrmse) if per_feature_nrmse else np.nan,
    })

horizon_metrics_df = pd.DataFrame(horizon_rows)
display(horizon_metrics_df.round(4))

plt.figure(figsize=(9, 4.5))
plt.plot(horizon_metrics_df["forecast_hour"], horizon_metrics_df["macro_normalized_MAE_by_training_IQR"], marker="o")
plt.xlabel("Forecast lead time (hours ahead)")
plt.ylabel("Macro-normalized MAE")
plt.title("PatchTST — Error by Forecast Horizon")
plt.xticks(range(1, HORIZON + 1, 2))
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("patchtst_error_by_forecast_horizon.png", dpi=160)
plt.show()

## 14. Actual vs predicted example — PM2.5

In [ ]:
pm25_idx = FEATURES.index("PM2.5 (µg/m³)")
# Prefer a complete 24-hour PM2.5 target window; otherwise use the window with most observed PM2.5 readings.
pm25_counts = test_mask[:, :, pm25_idx].sum(axis=1)
example_idx = int(np.argmax(pm25_counts))
actual_line = true_original[example_idx, :, pm25_idx].astype(float).copy()
actual_line[~test_mask[example_idx, :, pm25_idx]] = np.nan
pred_line = pred_original[example_idx, :, pm25_idx]

plt.figure(figsize=(10, 4.5))
plt.plot(np.arange(1, HORIZON + 1), actual_line, marker="o", label="Actual PM2.5")
plt.plot(np.arange(1, HORIZON + 1), pred_line, marker="o", label="PatchTST forecast")
plt.xlabel("Forecast lead time (hours ahead)")
plt.ylabel("PM2.5 (µg/m³)")
plt.title(f"PatchTST — 24-hour PM2.5 forecast example ({int(pm25_counts[example_idx])}/24 actual readings available)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("patchtst_pm25_actual_vs_predicted_example.png", dpi=160)
plt.show()

## 15. Inference speed and model size

In [ ]:
@torch.no_grad()
def benchmark_inference(model, input_shape, batch_size, warmup=10, repetitions=50):
    model.eval()
    x = torch.zeros((batch_size, *input_shape), dtype=torch.float32, device=DEVICE)
    for _ in range(warmup):
        _ = model(x)
    sync_device()
    start = time.perf_counter()
    for _ in range(repetitions):
        _ = model(x)
    sync_device()
    elapsed = time.perf_counter() - start
    ms_per_batch = (elapsed / repetitions) * 1000
    return {
        "batch_size": batch_size,
        "milliseconds_per_batch": ms_per_batch,
        "milliseconds_per_forecast": ms_per_batch / batch_size,
        "forecasts_per_second": 1000 / (ms_per_batch / batch_size),
    }

latency_batch1 = benchmark_inference(model, (SEQ_LEN, len(FEATURES)), batch_size=1)
latency_batch64 = benchmark_inference(model, (SEQ_LEN, len(FEATURES)), batch_size=64)
latency_df = pd.DataFrame([latency_batch1, latency_batch64])
display(latency_df.round(4))

print("Note: inference times depend on the Colab GPU/CPU type. Record the device name with the results.")

## 16. Save model, metrics, configuration and comparison-ready artifacts

In [ ]:
OUTPUT_DIR = Path("patchtst_artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

# Save the best validation checkpoint, not the final epoch by default.
checkpoint_path = OUTPUT_DIR / "patchtst_best_model.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "feature_names": FEATURES,
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "patch_len": PATCH_LEN,
    "patch_stride": PATCH_STRIDE,
    "num_patches": NUM_PATCHES,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "encoder_layers": E_LAYERS,
    "feedforward_dim": D_FF,
    "dropout": DROPOUT,
    "scaler_center": scaler.center_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "train_medians": train_medians.tolist(),
    "seed": SEED,
}, checkpoint_path)

checkpoint_size_mb = checkpoint_path.stat().st_size / (1024 ** 2)

metrics_df.to_csv(OUTPUT_DIR / "patchtst_test_metrics_by_pollutant.csv", index=False)
horizon_metrics_df.to_csv(OUTPUT_DIR / "patchtst_test_metrics_by_forecast_hour.csv", index=False)
history_df.to_csv(OUTPUT_DIR / "patchtst_training_history.csv", index=False)
latency_df.to_csv(OUTPUT_DIR / "patchtst_inference_benchmark.csv", index=False)

run_config = {
    **macro_summary,
    "model_checkpoint_size_MB": float(checkpoint_size_mb),
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seed": SEED,
    "features": FEATURES,
    "sequence_length_hours": SEQ_LEN,
    "forecast_horizon_hours": HORIZON,
    "architecture": "Channel-independent PatchTST",
    "patch_len": PATCH_LEN,
    "patch_stride": PATCH_STRIDE,
    "number_of_patches": NUM_PATCHES,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "encoder_layers": E_LAYERS,
    "feedforward_dim": D_FF,
    "split": {
        "train_fraction": TRAIN_FRACTION,
        "validation_fraction": VAL_FRACTION,
        "train_rows": int(train_end),
        "validation_rows": int(val_end-train_end),
        "test_rows": int(N-val_end),
        "train_start": str(train_start_time),
        "train_end": str(train_last_time),
        "validation_start": str(val_start_time),
        "validation_end": str(val_last_time),
        "test_start": str(test_start_time),
        "test_end": str(test_last_time),
    },
    "preprocessing": "Training-period median imputation for inputs; RobustScaler fit on imputed training period only; missing future targets masked in loss and metrics.",
    "best_validation_masked_mse": float(best_val_loss),
    "inference_benchmark_batch_1": latency_batch1,
    "inference_benchmark_batch_64": latency_batch64,
}
with open(OUTPUT_DIR / "patchtst_run_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

# Copy plots created above into the artifact folder.
for plot_name in [
    "patchtst_training_validation_loss.png",
    "patchtst_error_by_forecast_horizon.png",
    "patchtst_pm25_actual_vs_predicted_example.png",
]:
    if Path(plot_name).exists():
        import shutil
        shutil.copy2(plot_name, OUTPUT_DIR / plot_name)

zip_path = Path("patchtst_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in OUTPUT_DIR.iterdir():
        zf.write(item, arcname=item.name)

print("Saved checkpoint:", checkpoint_path)
print("Saved comparison tables, run summary and plots in:", OUTPUT_DIR.resolve())
print("Created:", zip_path.resolve())

## 17. What to report to the team

Share these items with the other three members so every model can be compared consistently:

1. `patchtst_test_metrics_by_pollutant.csv` — MAE, RMSE and R² for each pollutant in original units.
2. `patchtst_test_metrics_by_forecast_hour.csv` — error as forecast lead time increases.
3. `patchtst_run_summary.json` — split boundaries, macro-normalised scores, training time, parameter count, device and inference speed.
4. `patchtst_best_model.pt` — best checkpoint selected by validation masked MSE.
5. `patchtst_artifacts.zip` — all outputs bundled together.

### Fair-comparison rules for all eight base models

- Use the same 11 input/output pollutant columns, hourly rows, split boundaries, sequence length and horizon.
- Fit every imputer/scaler on training rows only. Do not fit on validation or test data.
- Missing future target readings must be excluded from the loss and metrics, not replaced by zero and scored as if measured.
- Report pollutant-wise metrics. Because pollutant units/ranges differ, use macro-normalised MAE/RMSE (normalised by each feature's training IQR) for a single cross-pollutant summary.
- Record training time, inference latency, parameter count, model/checkpoint size, peak GPU memory and the hardware used.
- Do not tune hyperparameters against the test period. Use validation results for model selection; evaluate the test set once the configuration is fixed.

**Next step:** run this notebook in Colab and share the resulting `patchtst_artifacts.zip` with the team. The TCN and PatchTST runs should use this same data pipeline and metric definitions; only the model architecture and its model-specific hyperparameters differ.
